## PX1 Mini-Cam — этап 6: поиск основного ПО и сценариев запуска

Запускается на твоём **Google Colab** с Android. По уже подтверждённому ARM/Linux-разделу ищет **только названия файлов** в вероятных каталогах управления и проверяет, какие известные названия служб упоминаются в системных сценариях запуска. Полный текст сценариев, секреты и пользовательские данные **не включаются в отчёт**. Исполняемые программы из образа не запускаются. Оригинальный `ЛП.imgc` только читается.

Нажми **Среда выполнения → Выполнить все**. Отчёт появится на твоём Google Диске: `PX1_SD_APP_DISCOVERY_REPORT.txt`. Если снова возникнет `CloseEvent`, сначала проверь наличие этого файла: обработка в Colab иногда успевает закончиться несмотря на обрыв соединения браузера.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, struct, shutil
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive')
src = root / 'Untitled folder' / 'ЛП.imgc'
if not src.is_file():
    src = next((Path(d)/'ЛП.imgc' for d,_,names in os.walk(root) if 'ЛП.imgc' in names), None)
if not src or not src.is_file(): raise FileNotFoundError('Не найден ЛП.imgc')
project=Path('/content/unimgc')
if not project.is_dir():
    subprocess.run(['git','clone','--depth','1','https://github.com/Shizmob/unimgc.git',str(project)],check=True)
subprocess.run(['make','-C',str(project)],check=True)
part=Path('/content/px1_ro_partition.img')
offset=32901120
size=7912719360
if shutil.disk_usage('/content').free < size + 1024**3:
    raise RuntimeError('Мало свободного места в Colab для копии раздела')
print('Читаю раздел в временный файл Colab (примерно 7.9 ГБ)...')
p=subprocess.Popen([str(project/'unimgc'),str(src)],stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
try:
    toskip=offset
    while toskip:
        data=p.stdout.read(min(4*1024*1024,toskip))
        if not data: raise EOFError('Образ закончился до начала раздела')
        toskip-=len(data)
    left=size
    with part.open('wb') as out:
        while left:
            data=p.stdout.read(min(4*1024*1024,left))
            if not data: raise EOFError('Образ закончился внутри раздела')
            out.write(data)
            left-=len(data)
finally:
    p.stdout.close()
    p.terminate()
    try: p.wait(timeout=10)
    except subprocess.TimeoutExpired: p.kill(); p.wait()
def dbg(command):
    v=subprocess.run(['debugfs','-R',command,str(part)],capture_output=True,timeout=90)
    return v.stdout.decode('utf-8','replace')
report=['PX1 Mini-Cam — офлайн-анализ системной информации Linux-раздела', 'Метка раздела по предыдущей проверке: linux_ltib']
for folder in ['/','/boot','/lib/modules','/etc','/opt','/usr/local']:
    raw=dbg('ls -p '+folder)
    names=[]
    for item in raw.splitlines():
        parts=item.strip().split('/')
        if len(parts)>=6 and parts[5]: names.append(parts[5])
    report.append('Каталог '+folder+' (до 35 имён): '+(', '.join(names[:35]) if names else 'нет данных'))
for f in ['/etc/os-release','/usr/lib/os-release','/etc/issue','/etc/ltib-release','/etc/debian_version']:
    raw=dbg('cat '+f)
    if raw and len(raw)<12000 and all(c.isprintable() or c in '\n\r\t' for c in raw):
        report.append(f+' (до 15 строк):\n'+'\n'.join(raw.splitlines()[:15]))
arch={3:'x86',40:'ARM',62:'x86-64',183:'AArch64',8:'MIPS',20:'PowerPC',4:'m68k'}
for f in ['/bin/busybox','/sbin/init','/bin/ls','/usr/bin/env']:
    dst=Path('/content/px1_elf_probe')
    if dst.exists(): dst.unlink()
    dbg('dump '+f+' '+str(dst))
    if dst.is_file():
        header=dst.open('rb').read(64)
        dst.unlink()
        if header[:4]==b'\x7fELF' and header[5] in (1,2):
            order='<' if header[5]==1 else '>'
            machine=struct.unpack_from(order+'H',header,18)[0]
            report.append('Архитектура ELF '+f+': '+arch.get(machine,'код '+str(machine)))
            break
for folder in ['/usr/local/bin','/usr/local/sbin','/usr/local/Trolltech','/etc/rc.d','/etc/init.d','/boot/u-boot','/ltib']:
    raw=dbg('ls -p '+folder)
    names=[]
    for line in raw.splitlines():
        fields=line.strip().split('/')
        if len(fields)>=6 and fields[5]: names.append(fields[5])
    report.append('Файлы '+folder+' (до 70 имён): '+(', '.join(names[:70]) if names else 'нет данных'))
config=dbg('cat /boot/linux.config')
for line in config.splitlines():
    if line.startswith(('CONFIG_INET=', 'CONFIG_IPV6=', 'CONFIG_MODULES=', 'CONFIG_SMB_FS=', 'CONFIG_CIFS=', 'CONFIG_WIRELESS=', 'CONFIG_USB_GADGET=', 'CONFIG_DEBUG_FS=', 'CONFIG_SECCOMP=')):
        report.append('Настройка ядра: '+line[:150])
# Metadata-only inspection of selected system paths. No file contents are copied into the report.
paths = [
    '/home', '/usr/bin', '/usr/sbin', '/sbin', '/bin',
    '/usr/local/lib', '/usr/local/share', '/etc/rc.d/init.d',
    '/usr/local/Trolltech/Qt-4.7.4-mx5x-x11/bin',
    '/usr/local/Trolltech/Qt-4.7.4-mx5x-x11/lib',
    '/usr/local/Trolltech/Qt-4.7.4-mx5x-x11/plugins',
    '/usr/local/Trolltech/Qt-4.7.4-mx5x-x11/plugins/imageformats',
    '/usr/local/Trolltech/Qt-4.7.2-mx5x-x11/lib',
    '/usr/local/Trolltech/Qt-4.7.2-mx5x-x11/plugins/imageformats',
    '/usr/local/Trolltech/Qtmobile-1.2.0-mx5x/bin',
    '/usr/local/Trolltech/Qtmobile-1.2.0-mx5x/lib'
]
for folder in paths:
    raw = dbg('ls -p '+folder)
    filenames = []
    for row in raw.splitlines():
        pieces = row.strip().split('/')
        if len(pieces)>=6 and pieces[5] and pieces[5] not in ('.','..'):
            filenames.append(pieces[5])
    report.append('Доп. каталог '+folder+' (до 100 названий): '+(', '.join(filenames[:100]) if filenames else 'нет данных'))

# Read-only targeted application discovery; publish metadata only.
for folder in ['/home/user','/home/revo','/root','/var/www','/var/www/html','/mnt','/usr/local/share','/etc/rc.d/init.d']:
    response=dbg('ls -p '+folder)
    found=[]
    for row in response.splitlines():
        items=row.strip().split('/')
        if len(items)>=6 and items[5] and items[5] not in ('.','..'): found.append(items[5])
    report.append('Поиск приложения '+folder+': '+(', '.join(found[:125]) if found else 'нет данных'))
import re
service_tokens=['dropbear','sshd','smbd','nmbd','telnetd','httpd','boa','inetd','dhcpd','wpa_supplicant','revo','minicam','proteus','ccu']
scripts=['/etc/inittab','/etc/rc.d/rcS','/etc/rc.d/rc.local','/etc/rc.d/rc.conf','/etc/rc.d/init.d/network','/etc/rc.d/init.d/sshd','/etc/rc.d/init.d/dropbear','/etc/rc.d/init.d/smb','/etc/rc.d/init.d/boa']
for name in scripts:
    raw=dbg('cat '+name)
    if not raw or len(raw)>180000: continue
    active=[line for line in raw.splitlines() if line.strip() and not line.lstrip().startswith('#')]
    hits=[key for key in service_tokens if any(re.search(r'(?<![a-z_])'+re.escape(key)+r'(?![a-z_])',ln,re.I) for ln in active)]
    report.append('Сценарий '+name+': активных строк='+str(len(active))+', совпадений по службам='+(','.join(hits) if hits else 'нет'))
report.append('Наличие сценария/упоминания службы не доказывает её запуск на устройстве.')
report.append('Подтверждённых уязвимостей этот отчёт не устанавливает.')
out=root/'PX1_SD_APP_DISCOVERY_REPORT.txt'
out.write_text('\n'.join(report)+'\n',encoding='utf-8')
part.unlink(missing_ok=True)
print('\n'.join(report))
print('Готово! Отчёт сохранён на Google Диске:',out.name)
